## Choosing a baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/baselines.ipynb)

Every occlusion-based method, WinTSR included, needs a **baseline**: the value an
occluded region is replaced with while the rest of the input stays real. Zero is the
default, and it's fine when zero is not itself a meaningful value in your data (e.g.
standardized inputs). When it *is* meaningful, occluding with zero looks like a signal
to the model, and attributions get noisy. This notebook shows the difference.

No dataset download needed.

In [ ]:
%pip install -q tslens matplotlib
# From source instead:
# %pip install -q "git+https://github.com/khairulislam/tslens.git"

## 1. A series where zero is a real value, not "nothing"

Every channel has a non-zero mean, so an occluded region set to zero isn't "no signal"
to this model — it's a value the channel never actually takes. The target still depends
on only **feature 0, steps 20-26**, same window as the quickstart.

In [2]:
import torch
from torch import nn

torch.manual_seed(0)
SEQ_LEN, N_FEATURES = 50, 5
SIGNAL_FEATURE, SIGNAL_START, SIGNAL_END = 0, 20, 26
FEATURE_MEANS = torch.tensor([6.0, -4.0, 3.0, 5.0, -2.0])


def make(n):
    x = torch.randn(n, SEQ_LEN, N_FEATURES) + FEATURE_MEANS
    # centered around the feature's own mean, so the target itself isn't just
    # "learn a big constant" -- the model still has to find the right window
    y = (x[:, SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE] - FEATURE_MEANS[SIGNAL_FEATURE]).sum(
        dim=1, keepdim=True
    )
    return x, y


x_train, y_train = make(4000)
x_test, y_test = make(256)
print("feature means (train):", x_train.mean(dim=(0, 1)).round(decimals=1).tolist())

feature means (train): [6.0, -4.0, 3.0, 5.0, -2.0]


## 2. Train a small GRU

In [3]:
class GRUForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(N_FEATURES, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out.mean(dim=1))


model = GRUForecaster()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.MSELoss()

for epoch in range(40):
    perm = torch.randperm(len(x_train))
    for i in range(0, len(x_train), 128):
        idx = perm[i : i + 128]
        opt.zero_grad()
        loss_fn(model(x_train[idx]), y_train[idx]).backward()
        opt.step()

model.eval()
with torch.no_grad():
    r2 = 1 - loss_fn(model(x_test), y_test).item() / y_test.var().item()
print(f"test R2 = {r2:.4f}")

test R2 = 0.9520


## 3. `get_baseline` — four ready-made options

| Mode | What it builds |
| --- | --- |
| `"zero"` | Zeros (the default). |
| `"random"` | Standard normal noise. |
| `"normal"` | Sampled from each feature's own mean/std. |
| `"mean"` | Each feature's mean, broadcast to every position. |

In [ ]:
from tslens import get_baseline

inputs = x_test[:64]
for mode in ["zero", "random", "normal", "mean"]:
    b = get_baseline(inputs, mode)
    print(f"{mode:<8} mean={b.mean(dim=(0, 1)).round(decimals=1).tolist()}")

## 4. Compare attribution quality against the known window

Average precision against the ground-truth mask (feature 0, steps 20-25). `"zero"`
occludes into a region the model never saw in training; `"normal"` and `"mean"` occlude
with values the model actually treats as unremarkable, so they should isolate the real
signal better.

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score
from tslens import WinTSR

ground_truth = torch.zeros(SEQ_LEN, N_FEATURES)
ground_truth[SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE] = 1


def average_precision(attr):
    a = attr.abs().mean(dim=1) if attr.dim() == 4 else attr.abs()
    rows = a.reshape(len(a), -1).detach().numpy()
    truth = ground_truth.reshape(-1).numpy()
    return float(np.mean([average_precision_score(truth, r) for r in rows]))


print(f"{'baseline':<10}{'avg precision':>14}")
print("-" * 24)
for mode in ["zero", "random", "normal", "mean"]:
    baselines = get_baseline(inputs, mode)
    attr = WinTSR(model).attribute(inputs, baselines=baselines, threshold=0.5)
    print(f"{mode:<10}{average_precision(attr):>14.4f}")

## 5. A custom baseline

`baselines` just needs to be a tensor the same shape as `inputs` (or a scalar). Anything
domain-specific — last observed value, a seasonal average, a different sample from the
dataset — works the same way.

In [6]:
last_step_baseline = inputs[:, -1:, :].expand_as(inputs)
attr = WinTSR(model).attribute(inputs, baselines=last_step_baseline, threshold=0.5)
print(f"custom (last-step) baseline avg precision: {average_precision(attr):.4f}")

custom (last-step) baseline avg precision: 0.8975


## Next steps

- **Rule of thumb.** Use `"mean"` or `"normal"` whenever zero is a real, in-distribution
  value for your data. Use `"zero"` for already-standardized inputs.
- **Tuples.** `get_baseline` recurses over tuples, so it works for multi-input models
  too — see the [TSlib models](tslib_models.ipynb) notebook.

Full recipe: [Choosing a baseline](https://khairulislam.github.io/tslens/integration/#choosing-a-baseline)
in the integration cookbook.